In [1]:
#shree ganesh ganpati bappa morya 💐🕉️

In [2]:
print("#shree ganesh ganpati bappa morya 💐🕉️")

#shree ganesh ganpati bappa morya 💐🕉️


In [3]:
# ============================================================
# CELL 1 — DENTAL AGE ESTIMATION PIPELINE
# ============================================================

import os
import sys
import json
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# REPOSITORY
# ------------------------------------------------------------

REPO = "/kaggle/working/Restrictive-Hierarchical-Semantic-Segmentation"

# ------------------------------------------------------------
# U-NET — FINAL SELECTED CHECKPOINT
# ------------------------------------------------------------

UNET_PATH = (
    "/kaggle/input/datasets/bitraj09/"
    "unethireextend/unet_hier_extended/"
    "fold_4/best.pt"
)

# ------------------------------------------------------------
# TEST OPG
# ------------------------------------------------------------

IMAGE_PATH = (
    "/kaggle/input/datasets/bitraj09/"
    "unseen/panoramic (39).jpg"
)

# ------------------------------------------------------------
# HIERARCHY
# ------------------------------------------------------------

HIERARCHY_PATH = os.path.join(
    REPO,
    "class_tree_tl_extended.json"
)

# ------------------------------------------------------------
# CLASS MAP
# ------------------------------------------------------------

CLASS_MAP_PATH = os.path.join(
    REPO,
    "class_map_extended.csv"
)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# TARGET FDI TEETH FOR AGE ESTIMATION
# ------------------------------------------------------------

TARGET_FDI = [13, 23, 33, 43]

# ------------------------------------------------------------
# CHECK EVERYTHING
# ------------------------------------------------------------

print("=" * 80)
print("DENTAL AGE ESTIMATION — PIPELINE INITIALIZATION")
print("=" * 80)

print("\nRepository:")
print(REPO)

print("\nU-Net:")
print(UNET_PATH)
print("Exists:", os.path.exists(UNET_PATH))

print("\nTest OPG:")
print(IMAGE_PATH)
print("Exists:", os.path.exists(IMAGE_PATH))

print("\nHierarchy:")
print(HIERARCHY_PATH)
print("Exists:", os.path.exists(HIERARCHY_PATH))

print("\nClass map:")
print(CLASS_MAP_PATH)
print("Exists:", os.path.exists(CLASS_MAP_PATH))

print("\nDevice:", DEVICE)

print("\nTarget FDI teeth:", TARGET_FDI)

print("\n" + "=" * 80)
print("✓ CELL 1 COMPLETE")
print("=" * 80)

DENTAL AGE ESTIMATION — PIPELINE INITIALIZATION

Repository:
/kaggle/working/Restrictive-Hierarchical-Semantic-Segmentation

U-Net:
/kaggle/input/datasets/bitraj09/unethireextend/unet_hier_extended/fold_4/best.pt
Exists: True

Test OPG:
/kaggle/input/datasets/bitraj09/unseen/panoramic (39).jpg
Exists: True

Hierarchy:
/kaggle/working/Restrictive-Hierarchical-Semantic-Segmentation/class_tree_tl_extended.json
Exists: False

Class map:
/kaggle/working/Restrictive-Hierarchical-Semantic-Segmentation/class_map_extended.csv
Exists: False

Device: cuda

Target FDI teeth: [13, 23, 33, 43]

✓ CELL 1 COMPLETE


In [4]:
# ============================================================
# CELL 2 — PREPARE REPOSITORY + VERIFIED METADATA
# ============================================================

import os
import json
import pandas as pd
import numpy as np

print("=" * 80)
print("PREPARING MODEL METADATA")
print("=" * 80)

# ------------------------------------------------------------
# CHECK REPOSITORY
# ------------------------------------------------------------

print("REPO:")
print(REPO)
print("Exists:", os.path.exists(REPO))

if not os.path.exists(REPO):
    os.makedirs(REPO, exist_ok=True)
    print("✓ Repository directory created")

# ------------------------------------------------------------
# VERIFIED HIERARCHY
# ------------------------------------------------------------

class_tree = {
    "background": {},
    "tooth+alveolar": {
        "alveolar": {
            "upper": {},
            "lower": {}
        },
        "tooth": {
            "composite": {},
            "healthy": {
                "pulp": {},
                "dentin": {},
                "enamel": {}
            }
        }
    }
}

HIERARCHY_PATH = os.path.join(
    REPO,
    "class_tree_tl_extended.json"
)

with open(HIERARCHY_PATH, "w") as f:
    json.dump(class_tree, f, indent=2)

# ------------------------------------------------------------
# VERIFIED CLASS MAP
# ------------------------------------------------------------

class_map = pd.DataFrame({
    "class_id": range(11),

    "class_name": [
        "background",
        "tooth+alveolar",
        "alveolar",
        "tooth",
        "upper",
        "lower",
        "composite",
        "healthy",
        "pulp",
        "dentin",
        "enamel"
    ],

    "pixel_val": [
        0.0,
        np.nan,
        np.nan,
        np.nan,
        212.0,
        255.0,
        42.0,
        np.nan,
        127.0,
        170.0,
        85.0
    ]
})

CLASS_MAP_PATH = os.path.join(
    REPO,
    "class_map_extended.csv"
)

class_map.to_csv(
    CLASS_MAP_PATH,
    index=False
)

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("\nHierarchy:")
print(HIERARCHY_PATH)
print("Exists:", os.path.exists(HIERARCHY_PATH))

print("\nClass map:")
print(CLASS_MAP_PATH)
print("Exists:", os.path.exists(CLASS_MAP_PATH))

print("\n" + "=" * 80)
print("CLASS MAP")
print("=" * 80)

print(class_map.to_string(index=False))

print("\n" + "=" * 80)
print("✓ CELL 2 COMPLETE")
print("=" * 80)

PREPARING MODEL METADATA
REPO:
/kaggle/working/Restrictive-Hierarchical-Semantic-Segmentation
Exists: False
✓ Repository directory created

Hierarchy:
/kaggle/working/Restrictive-Hierarchical-Semantic-Segmentation/class_tree_tl_extended.json
Exists: True

Class map:
/kaggle/working/Restrictive-Hierarchical-Semantic-Segmentation/class_map_extended.csv
Exists: True

CLASS MAP
 class_id     class_name  pixel_val
        0     background        0.0
        1 tooth+alveolar        NaN
        2       alveolar        NaN
        3          tooth        NaN
        4          upper      212.0
        5          lower      255.0
        6      composite       42.0
        7        healthy        NaN
        8           pulp      127.0
        9         dentin      170.0
       10         enamel       85.0

✓ CELL 2 COMPLETE


In [5]:
# ============================================================
# CELL 3 — LOAD FINAL U-NET FOLD 4
# ============================================================

import sys
import torch

# ------------------------------------------------------------
# AUTHOR REPOSITORY
# ------------------------------------------------------------

if REPO not in sys.path:
    sys.path.insert(0, REPO)

from Models import models

print("=" * 80)
print("CREATING FINAL U-NET — FOLD 4")
print("=" * 80)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# ------------------------------------------------------------
# CREATE EXACT MODEL USED IN COMPARISON NOTEBOOK
# ------------------------------------------------------------

model = models.UNet(
    size=620,
    n_channels=3,
    hierarchy=class_tree,
    model_type=1
)

model = model.to(DEVICE)

print("✓ U-Net created")

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

# ------------------------------------------------------------
# LOAD FOLD 4 CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    UNET_PATH,
    map_location=DEVICE,
    weights_only=False
)

print("\nCheckpoint:")
print("Epoch:", checkpoint["epoch"])
print("Loss :", checkpoint["loss"])
print("Test mean:", checkpoint["test_measure_mean"])
print("Test std :", checkpoint["test_measure_std"])

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("\n" + "=" * 80)
print("✓ FINAL FOLD 4 U-NET READY")
print("=" * 80)

ModuleNotFoundError: No module named 'Models'

In [ ]:
# ============================================================
# CELL 4 — FOLD 4 U-NET PREDICTION
# ============================================================

import cv2
import numpy as np
import torch

print("=" * 80)
print("FOLD 4 — SINGLE OPG PREDICTION")
print("=" * 80)

# ------------------------------------------------------------
# LOAD OPG
# ------------------------------------------------------------

image_bgr = cv2.imread(IMAGE_PATH)

if image_bgr is None:
    raise FileNotFoundError(
        f"Image not found:\n{IMAGE_PATH}"
    )

image_rgb = cv2.cvtColor(
    image_bgr,
    cv2.COLOR_BGR2RGB
)

original_image = image_rgb.copy()

print("Original image shape:", image_rgb.shape)

# ------------------------------------------------------------
# RESIZE TO AUTHOR MODEL INPUT
# ------------------------------------------------------------

input_rgb = cv2.resize(
    image_rgb,
    (620, 620),
    interpolation=cv2.INTER_LINEAR
)

# ------------------------------------------------------------
# TO TENSOR
# ------------------------------------------------------------

x = input_rgb.astype(np.float32) / 255.0

x = torch.from_numpy(x)
x = x.permute(2, 0, 1)
x = x.unsqueeze(0)
x = x.to(DEVICE)

print("Model input:", tuple(x.shape))

# ------------------------------------------------------------
# INFERENCE
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    output = model(
        x,
        type=1,
        hierarchy=class_tree
    )

print("\n✓ Prediction completed")

# ------------------------------------------------------------
# OUTPUT STRUCTURE
# ------------------------------------------------------------

print("\nOutput type:", type(output))
print("Number of outputs:", len(output))

for i, level_output in enumerate(output):

    print(
        f"\nOutput {i}:",
        type(level_output),
        "length:",
        len(level_output)
    )

    for j, tensor in enumerate(level_output):

        print(
            f"  Level {j}:",
            tuple(tensor.shape),
            "min:",
            float(tensor.min()),
            "max:",
            float(tensor.max())
        )

print("\n" + "=" * 80)
print("✓ CELL 4 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 5 — FINAL AUTHOR-STYLE SEGMENTATION
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("=" * 80)
print("BUILDING FINAL SEMANTIC SEGMENTATION")
print("=" * 80)

# ------------------------------------------------------------
# PROBABILITY OUTPUTS
# ------------------------------------------------------------

p0 = output[0][0][0].detach().cpu().numpy()  # 2
p1 = output[0][1][0].detach().cpu().numpy()  # 2
p2 = output[0][2][0].detach().cpu().numpy()  # 4
p3 = output[0][3][0].detach().cpu().numpy()  # 3

# ------------------------------------------------------------
# ARGMAX PER LEVEL
# ------------------------------------------------------------

l0 = np.argmax(p0, axis=0)
l1 = np.argmax(p1, axis=0)
l2 = np.argmax(p2, axis=0)
l3 = np.argmax(p3, axis=0)

# ------------------------------------------------------------
# FINAL SEMANTIC CLASS MAP
#
# 0  background
# 1  tooth+alveolar
# 2  alveolar
# 3  tooth
# 4  upper
# 5  lower
# 6  composite
# 7  healthy
# 8  pulp
# 9  dentin
# 10 enamel
# ------------------------------------------------------------

semantic = np.zeros_like(l0, dtype=np.uint8)

foreground = (l0 == 1)

# Level 1
alveolar = foreground & (l1 == 0)
tooth     = foreground & (l1 == 1)

semantic[alveolar] = 2
semantic[tooth] = 3

# Level 2
upper     = alveolar & (l2 == 0)
lower     = alveolar & (l2 == 1)
composite = tooth & (l2 == 2)
healthy   = tooth & (l2 == 3)

semantic[upper]     = 4
semantic[lower]     = 5
semantic[composite] = 6
semantic[healthy]   = 7

# Level 3
pulp   = healthy & (l3 == 0)
dentin = healthy & (l3 == 1)
enamel = healthy & (l3 == 2)

semantic[pulp]   = 8
semantic[dentin] = 9
semantic[enamel] = 10

# ------------------------------------------------------------
# PIXEL SUMMARY
# ------------------------------------------------------------

class_names = {
    4: "Upper",
    5: "Lower",
    6: "Composite",
    8: "Pulp",
    9: "Dentin",
    10: "Enamel",
}

print("\nSEMANTIC PIXELS")
print("-" * 50)

for cls_id, name in class_names.items():
    print(
        f"{name:10s}: "
        f"{int(np.sum(semantic == cls_id)):,}"
    )

# ------------------------------------------------------------
# DRAW CONTOURS ON ORIGINAL 620x620 INPUT
# ------------------------------------------------------------

visual = input_rgb.copy()

COLORS = {
    4: (255, 255, 0),    # upper
    5: (255, 0, 255),    # lower
    6: (0, 255, 0),      # composite
    8: (0, 0, 255),      # pulp
    9: (255, 255, 255),  # dentin
    10: (255, 0, 0),     # enamel
}

for cls_id, color in COLORS.items():

    mask = (
        (semantic == cls_id)
        .astype(np.uint8)
        * 255
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    cv2.drawContours(
        visual,
        contours,
        -1,
        color,
        1
    )

# ------------------------------------------------------------
# TOOTH OUTER BOUNDARY
# ------------------------------------------------------------

tooth_mask = (
    (tooth | composite | healthy)
    .astype(np.uint8)
    * 255
)

contours, _ = cv2.findContours(
    tooth_mask,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

cv2.drawContours(
    visual,
    contours,
    -1,
    (0, 255, 0),
    1
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(22, 8)
)

axes[0].imshow(input_rgb)
axes[0].set_title("Original OPG", fontsize=16)
axes[0].axis("off")

axes[1].imshow(visual)
axes[1].set_title(
    "Fold 4 — Hierarchical Segmentation",
    fontsize=16
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("✓ CELL 5 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 6 — LOAD AUTHOR MASK R-CNN IMPLEMENTATION
# ============================================================

import os
import sys
import subprocess

MASKRCNN_REPO = (
    "/kaggle/working/dental-segmentation"
)

# ------------------------------------------------------------
# Clone repository
# ------------------------------------------------------------

if not os.path.exists(MASKRCNN_REPO):

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/dorpetrovic/dental-segmentation.git",
            MASKRCNN_REPO
        ],
        check=True
    )

print("=" * 80)
print("MASK R-CNN REPOSITORY")
print("=" * 80)

print("Repository:", MASKRCNN_REPO)

# ------------------------------------------------------------
# Verify author's model implementation
# ------------------------------------------------------------

MODEL_FILE = os.path.join(
    MASKRCNN_REPO,
    "models",
    "teeth_segmentation.py"
)

CONFIG_FILE = os.path.join(
    MASKRCNN_REPO,
    "configs",
    "model_config.py"
)

print("\nModel implementation:")
print(MODEL_FILE)
print("Exists:", os.path.exists(MODEL_FILE))

print("\nConfiguration:")
print(CONFIG_FILE)
print("Exists:", os.path.exists(CONFIG_FILE))

# ------------------------------------------------------------
# Import author's implementation
# ------------------------------------------------------------

if MASKRCNN_REPO not in sys.path:
    sys.path.insert(0, MASKRCNN_REPO)

print("\n" + "=" * 80)
print("✓ MASK R-CNN REPOSITORY READY")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 7 — INSPECT AUTHOR MASK R-CNN IMPLEMENTATION
# ============================================================

import os

print("=" * 80)
print("SEARCHING AUTHOR MASK R-CNN FILES")
print("=" * 80)

for root, dirs, files in os.walk(MASKRCNN_REPO):

    for file in files:

        if file.endswith(".py"):

            path = os.path.join(root, file)

            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()

                keywords = [
                    "maskrcnn_resnet50_fpn",
                    "MaskRCNN",
                    "build_model",
                    "load_inference_model",
                    "best.pth",
                    "num_classes",
                    "FastRCNNPredictor",
                    "MaskRCNNPredictor"
                ]

                if any(k in text for k in keywords):

                    print("\n" + "-" * 80)
                    print(path)
                    print("-" * 80)

                    for i, line in enumerate(text.splitlines(), 1):

                        if any(k.lower() in line.lower() for k in keywords):

                            print(f"{i:4d}: {line.strip()}")

            except Exception:
                pass

print("\n" + "=" * 80)
print("✓ CELL 7 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 8 — DOWNLOAD OFFICIAL MASK R-CNN WEIGHTS + LOAD
# ============================================================

import os
import torch

from huggingface_hub import hf_hub_download

from models.teeth_segmentation import (
    build_model,
    BINARY,
)

print("=" * 80)
print("LOADING OFFICIAL MASK R-CNN CHECKPOINT")
print("=" * 80)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

NUM_CLASSES = 2 if BINARY else 36

WEIGHTS_DIR = (
    "/kaggle/working/dental-segmentation/"
    "outputs/results/maskrcnn_torch"
)

os.makedirs(
    WEIGHTS_DIR,
    exist_ok=True
)

TOOTH_WEIGHTS = os.path.join(
    WEIGHTS_DIR,
    "best.pth"
)

print("BINARY:", BINARY)
print("Classes:", NUM_CLASSES)
print("Target checkpoint:", TOOTH_WEIGHTS)

# ------------------------------------------------------------
# DOWNLOAD FROM HUGGING FACE IF MISSING
# ------------------------------------------------------------

if not os.path.exists(TOOTH_WEIGHTS):

    print("\nCheckpoint not found locally.")
    print("Downloading official best.pth from Hugging Face...")

    downloaded_path = hf_hub_download(
        repo_id="chocodo/dental-segmentation-maskrcnn-torch",
        filename="best.pth",
        local_dir=WEIGHTS_DIR,
    )

    print("\n✓ Download complete")
    print("Downloaded:", downloaded_path)

else:

    print("\n✓ Checkpoint already exists")

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

if not os.path.exists(TOOTH_WEIGHTS):
    raise FileNotFoundError(
        f"Checkpoint still not found:\n{TOOTH_WEIGHTS}"
    )

size_mb = (
    os.path.getsize(TOOTH_WEIGHTS)
    / (1024 ** 2)
)

print(
    f"Checkpoint size: {size_mb:.2f} MB"
)

# ------------------------------------------------------------
# BUILD AUTHOR MODEL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BUILDING AUTHOR MASK R-CNN")
print("=" * 80)

tooth_model = build_model(
    NUM_CLASSES
)

tooth_model = tooth_model.to(
    DEVICE
)

# ------------------------------------------------------------
# LOAD WEIGHTS
# ------------------------------------------------------------

checkpoint = torch.load(
    TOOTH_WEIGHTS,
    map_location=DEVICE,
    weights_only=True
)

tooth_model.load_state_dict(
    checkpoint,
    strict=True
)

tooth_model.eval()

print("\n" + "=" * 80)
print("✓ MASK R-CNN READY")
print("=" * 80)

print("Model:", type(tooth_model))
print("Classes:", NUM_CLASSES)
print("Device:", DEVICE)
print("Training:", tooth_model.training)

In [ ]:
# ============================================================
# CELL 9 — AUTHOR-CORRECT MASK R-CNN FDI PREDICTION
# ============================================================

import cv2
import torch
import numpy as np

from torchvision.transforms.functional import to_tensor
from utils.preprocessing import enhance_contrast
from configs.model_config import (
    CONF_THRESHOLD,
    FDI_CLASSES
)

print("=" * 80)
print("AUTHOR-STYLE MASK R-CNN FDI PREDICTION")
print("=" * 80)

# ------------------------------------------------------------
# LOAD IMAGE
# ------------------------------------------------------------

image_bgr = cv2.imread(IMAGE_PATH)

if image_bgr is None:
    raise FileNotFoundError(IMAGE_PATH)

image_rgb_full = cv2.cvtColor(
    image_bgr,
    cv2.COLOR_BGR2RGB
)

print("Original image:", image_rgb_full.shape)

# ------------------------------------------------------------
# AUTHOR PREPROCESSING
# CLAHE on LAB L-channel
# ------------------------------------------------------------

enhanced = enhance_contrast(
    image_rgb_full,
    method="clahe"
)

print("CLAHE preprocessing: ✓")

# ------------------------------------------------------------
# AUTHOR TENSOR CONVERSION
# torchvision handles resize internally
# ------------------------------------------------------------

image_tensor = to_tensor(
    enhanced
).to(DEVICE)

print(
    "Input tensor:",
    tuple(image_tensor.shape)
)

# ------------------------------------------------------------
# INFERENCE
# ------------------------------------------------------------

with torch.no_grad():

    prediction = tooth_model(
        [image_tensor]
    )[0]

# ------------------------------------------------------------
# RAW OUTPUT
# ------------------------------------------------------------

boxes = (
    prediction["boxes"]
    .detach()
    .cpu()
    .numpy()
)

scores = (
    prediction["scores"]
    .detach()
    .cpu()
    .numpy()
)

labels = (
    prediction["labels"]
    .detach()
    .cpu()
    .numpy()
)

masks = (
    prediction["masks"]
    .detach()
    .cpu()
    .numpy()
)

print("\nRaw detections:", len(labels))

# ------------------------------------------------------------
# AUTHOR CONFIDENCE THRESHOLD
# ------------------------------------------------------------

CONF_THRESHOLD = float(CONF_THRESHOLD)

print(
    "Confidence threshold:",
    CONF_THRESHOLD
)

keep = scores >= CONF_THRESHOLD

boxes = boxes[keep]
scores = scores[keep]
labels = labels[keep]
masks = masks[keep]

print(
    "Detections after threshold:",
    len(labels)
)

# ------------------------------------------------------------
# PRINT ALL FDI DETECTIONS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ALL FDI DETECTIONS")
print("=" * 80)

for i in range(len(labels)):

    class_id = int(labels[i])

    class_name = (
        FDI_CLASSES[class_id]
        if class_id < len(FDI_CLASSES)
        else str(class_id)
    )

    print(
        f"{i:3d} | "
        f"FDI={class_name:>3s} | "
        f"score={scores[i]:.4f} | "
        f"pixels={int((masks[i,0] > 0.5).sum())}"
    )

# ------------------------------------------------------------
# EXTRACT TARGET CANINES
# ------------------------------------------------------------

TARGET_FDI = {"13", "23", "33", "43"}

fdi_detections = {}

for i in range(len(labels)):

    class_id = int(labels[i])

    if class_id >= len(FDI_CLASSES):
        continue

    fdi = FDI_CLASSES[class_id]

    if fdi not in TARGET_FDI:
        continue

    score = float(scores[i])

    binary_mask = (
        masks[i, 0] > 0.5
    )

    # Keep highest-confidence duplicate
    if (
        fdi not in fdi_detections
        or score > fdi_detections[fdi]["score"]
    ):

        fdi_detections[fdi] = {
            "index": i,
            "score": score,
            "mask": binary_mask,
            "box": boxes[i].astype(int),
            "class_id": class_id
        }

# ------------------------------------------------------------
# TARGET SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TARGET CANINES")
print("=" * 80)

for fdi in ["13", "23", "33", "43"]:

    if fdi in fdi_detections:

        d = fdi_detections[fdi]

        print(
            f"FDI {fdi}: "
            f"FOUND | "
            f"score={d['score']:.4f} | "
            f"pixels={int(d['mask'].sum())} | "
            f"box={tuple(d['box'])}"
        )

    else:

        print(
            f"FDI {fdi}: NOT DETECTED"
        )

print("\n" + "=" * 80)
print("✓ CELL 9 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 10 — ALIGN U-NET MASKS TO ORIGINAL OPG
# ============================================================

import cv2
import numpy as np

print("=" * 80)
print("ALIGNING U-NET OUTPUT WITH ORIGINAL OPG")
print("=" * 80)

ORIG_H, ORIG_W = image_rgb_full.shape[:2]

print("Original OPG:", (ORIG_H, ORIG_W))
print("U-Net map   :", semantic.shape)

# ------------------------------------------------------------
# RESIZE FINAL SEMANTIC MAP TO ORIGINAL IMAGE SIZE
# IMPORTANT: nearest-neighbor preserves class IDs
# ------------------------------------------------------------

semantic_full = cv2.resize(
    semantic,
    (ORIG_W, ORIG_H),
    interpolation=cv2.INTER_NEAREST
)

print(
    "Aligned semantic map:",
    semantic_full.shape
)

# ------------------------------------------------------------
# CREATE SPECIFIC FULL-RES MASKS
# ------------------------------------------------------------

tooth_mask_full = np.isin(
    semantic_full,
    [3, 6, 7, 8, 9, 10]
)

pulp_mask_full = (
    semantic_full == 8
)

dentin_mask_full = (
    semantic_full == 9
)

enamel_mask_full = (
    semantic_full == 10
)

print("\nFull-resolution masks:")
print(
    "Tooth pixels:",
    int(tooth_mask_full.sum())
)

print(
    "Pulp pixels :",
    int(pulp_mask_full.sum())
)

print(
    "Dentin pixels:",
    int(dentin_mask_full.sum())
)

print(
    "Enamel pixels:",
    int(enamel_mask_full.sum())
)

# ------------------------------------------------------------
# CHECK MASK R-CNN DIMENSIONS
# ------------------------------------------------------------

sample_fdi = fdi_detections["13"]["mask"]

print("\nMask R-CNN FDI mask:")
print("FDI 13 shape:", sample_fdi.shape)

assert sample_fdi.shape == (ORIG_H, ORIG_W), (
    f"Unexpected Mask R-CNN mask shape: {sample_fdi.shape}"
)

print("\n✓ U-Net and Mask R-CNN are now in same coordinate space")

print("\n" + "=" * 80)
print("✓ CELL 10 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 11 — INDIVIDUAL CANINE PA/TA
# ============================================================

print("=" * 80)
print("INDIVIDUAL CANINE PULP / TOOTH RATIOS")
print("=" * 80)

age_measurements = {}

for fdi in ["13", "23", "33", "43"]:

    if fdi not in fdi_detections:
        print(f"FDI {fdi}: skipped — not detected")
        continue

    fdi_mask = fdi_detections[fdi]["mask"]

    # --------------------------------------------------------
    # TOOTH AREA
    # Use the Mask R-CNN individual FDI tooth mask
    # --------------------------------------------------------

    tooth_pixels = int(
        fdi_mask.sum()
    )

    # --------------------------------------------------------
    # PULP AREA
    # Intersection of:
    #   U-Net pulp
    #   AND
    #   FDI-specific tooth
    # --------------------------------------------------------

    fdi_pulp_mask = (
        fdi_mask &
        pulp_mask_full
    )

    pulp_pixels = int(
        fdi_pulp_mask.sum()
    )

    # --------------------------------------------------------
    # PA/TA
    # --------------------------------------------------------

    ratio = (
        pulp_pixels / tooth_pixels
        if tooth_pixels > 0
        else 0.0
    )

    ratio_percent = ratio * 100.0

    age_measurements[fdi] = {
        "FDI": int(fdi),
        "Tooth_pixels": tooth_pixels,
        "Pulp_pixels": pulp_pixels,
        "PA_TA": ratio,
        "PA_TA_percent": ratio_percent,
        "MaskRCNN_score": fdi_detections[fdi]["score"],
        "Pulp_mask": fdi_pulp_mask
    }

    print(
        f"\nFDI {fdi}"
    )

    print(
        f"  Tooth pixels : {tooth_pixels:,}"
    )

    print(
        f"  Pulp pixels  : {pulp_pixels:,}"
    )

    print(
        f"  PA/TA        : {ratio:.6f}"
    )

    print(
        f"  PA/TA (%)    : {ratio_percent:.3f}%"
    )

print("\n" + "=" * 80)
print("✓ CELL 11 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# CELL 12 — VISUALIZE FDI-SPECIFIC TOOTH + PULP
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

TARGETS = ["13", "23", "33", "43"]

for fdi in TARGETS:

    if fdi not in age_measurements:
        print(f"FDI {fdi}: not available")
        continue

    tooth_mask = fdi_detections[fdi]["mask"]
    pulp_mask = age_measurements[fdi]["Pulp_mask"]

    score = fdi_detections[fdi]["score"]

    # --------------------------------------------------------
    # FIND CROP FROM TOOTH MASK
    # --------------------------------------------------------

    ys, xs = np.where(tooth_mask)

    if len(xs) == 0:
        print(f"FDI {fdi}: empty mask")
        continue

    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()

    pad = 25

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(ORIG_W - 1, x2 + pad)
    y2 = min(ORIG_H - 1, y2 + pad)

    # --------------------------------------------------------
    # CROP ORIGINAL
    # --------------------------------------------------------

    crop = image_rgb_full[
        y1:y2 + 1,
        x1:x2 + 1
    ].copy()

    tooth_crop = tooth_mask[
        y1:y2 + 1,
        x1:x2 + 1
    ]

    pulp_crop = pulp_mask[
        y1:y2 + 1,
        x1:x2 + 1
    ]

    # --------------------------------------------------------
    # DRAW TOOTH + PULP CONTOURS
    # --------------------------------------------------------

    visual = crop.copy()

    # Tooth boundary = GREEN
    tooth_u8 = (
        tooth_crop.astype(np.uint8) * 255
    )

    tooth_contours, _ = cv2.findContours(
        tooth_u8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    cv2.drawContours(
        visual,
        tooth_contours,
        -1,
        (0, 255, 0),
        2
    )

    # Pulp boundary = BLUE
    pulp_u8 = (
        pulp_crop.astype(np.uint8) * 255
    )

    pulp_contours, _ = cv2.findContours(
        pulp_u8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    cv2.drawContours(
        visual,
        pulp_contours,
        -1,
        (0, 0, 255),
        2
    )

    # --------------------------------------------------------
    # OPTIONAL: LIGHT PULP FILL
    # --------------------------------------------------------

    pulp_overlay = visual.copy()

    pulp_overlay[pulp_crop] = (
        255, 0, 0
    )

    visual = cv2.addWeighted(
        visual,
        0.75,
        pulp_overlay,
        0.25,
        0
    )

    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    tooth_pixels = int(tooth_mask.sum())
    pulp_pixels = int(pulp_mask.sum())

    ratio = (
        pulp_pixels / tooth_pixels
        if tooth_pixels > 0
        else 0
    )

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5)
    )

    # Original crop
    axes[0].imshow(crop)
    axes[0].set_title(
        f"FDI {fdi} — Original"
    )
    axes[0].axis("off")

    # Tooth mask
    axes[1].imshow(
        tooth_crop,
        cmap="gray"
    )
    axes[1].set_title(
        f"Tooth Mask\n"
        f"{tooth_pixels:,} px"
    )
    axes[1].axis("off")

    # Combined
    axes[2].imshow(visual)
    axes[2].set_title(
        f"Tooth + Pulp\n"
        f"Pulp: {pulp_pixels:,} px\n"
        f"PA/TA: {ratio*100:.2f}%\n"
        f"Score: {score:.3f}"
    )
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

print("=" * 80)
print("✓ CELL 12 COMPLETE")
print("=" * 80)